# Finnhub free-tier example: NVIDIA (NVDA)

This notebook uses only Finnhub's free real-time US stock quote endpoint. It loads `FINNHUB_API_KEY` from the repository's `.env` file without displaying the key. Historical candles are intentionally excluded because Finnhub classifies them as premium.

## Setup

Run `uv sync --extra docs` from the repository root and select the project's virtual environment as the kernel.

In [1]:
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

from agentic_data_pipeline.ingestion import FinnhubClient

repo_root = Path.cwd()
if not (repo_root / 'pyproject.toml').exists():
    repo_root = repo_root.parent
env_path = repo_root / '.env'
if not env_path.is_file():
    raise FileNotFoundError(f'Expected API credentials at {env_path}')

load_dotenv(env_path)
client = FinnhubClient()

## Fetch NVIDIA's current quote

The result is a provider-independent `MarketQuote`. Its timestamp is timezone-aware UTC.

In [2]:
quote = client.get_quote('NVDA')
quote

MarketQuote(timestamp=datetime.datetime(2026, 8, 17, 20, 0, tzinfo=datetime.timezone.utc), symbol='NVDA', current_price=225.01, open=225.98, high=227.92, low=224.86, previous_close=225.16, change=-0.15, percent_change=-0.0666, source='finnhub')

## Use the standardized representation

`to_dict()` is suitable for JSON serialization or conversion to pandas. A quote contains the current price, session open/high/low, previous close, and price changes; it does not pretend to be a historical candle.

In [4]:
quote_frame = pd.DataFrame([quote.to_dict()]).set_index('timestamp')
display(quote_frame)

,symbol,current_price,open,high,low,previous_close,change,percent_change,source
timestamp,,,,,,,,,
2026-08-17T20:00:00Z,NVDA,225.01,225.98,227.92,224.86,225.16,-0.15,-0.0666,finnhub


## Calculate a few session indicators

These values are derived locally and do not make additional API requests. Finnhub recommends avoiding constant polling of the REST quote endpoint.

In [5]:
session_range = quote.high - quote.low
move_from_open = quote.current_price - quote.open

print(f'NVDA current price: ${quote.current_price:,.2f}')
print(f'Session range: ${session_range:,.2f}')
print(f'Move from open: ${move_from_open:,.2f}')
if quote.percent_change is not None:
    print(f'Change from previous close: {quote.percent_change:,.2f}%')

NVDA current price: $225.01
Session range: $3.06
Move from open: $-0.97
Change from previous close: -0.07%
